In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np
import ast
import optuna
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostRegressor, Pool
from sklearn.feature_selection import RFE
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.utils import shuffle
from scipy.stats import randint
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, make_scorer
from sklearn.model_selection import train_test_split, GridSearchCV, GroupShuffleSplit, RandomizedSearchCV, cross_val_score, KFold
from shapely.geometry import LineString, Point
from tqdm import tqdm
from itertools import combinations
from sklearn.neighbors import BallTree
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler, FunctionTransformer, PolynomialFeatures, RobustScaler, LabelEncoder
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.base import BaseEstimator, RegressorMixin
from xgboost import XGBRegressor, plot_importance, DMatrix, train
from lightgbm import LGBMRegressor, early_stopping, Dataset, plot_importance, log_evaluation
import shap
from itertools import product
from scipy.stats import zscore, uniform, randint
from scipy.spatial import cKDTree
import os
import fiona
import random
from geopy.distance import geodesic

/Users/silarbinunzio/opt/anaconda3/lib/python3.8/site-packages/dask/dataframe/utils.py:367: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  _numeric_index_types = (pd.Int64Index, pd.Float64Index, pd.UInt64Index)
/Users/silarbinunzio/opt/anaconda3/lib/python3.8/site-packages/dask/dataframe/utils.py:367: FutureWarning: pandas.Float64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  _numeric_index_types = (pd.Int64Index, pd.Float64Index, pd.UInt64Index)
/Users/silarbinunzio/opt/anaconda3/lib/python3.8/site-packages/dask/dataframe/utils.py:367: FutureWarning: pandas.UInt64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  _numeric_index_types = (pd.Int64Index, pd.Float64Index, pd.UInt64Index)


In [3]:
doubs_data = gpd.read_file("../datasets_par_departement/departement-25-doubs-original.geojson")
ain_data = gpd.read_file("../datasets_par_departement/departement-01-ain-original.geojson")
yvelines_data = gpd.read_file("../datasets_par_departement/departement-78-yvelines-original.geojson")
firepoint_1 = gpd.read_file("../data/hexagones_firepoint_1.geojson")
firepoint_2 = gpd.read_file("../data/hexagones_firepoint_2.geojson")
firepoint_3 = gpd.read_file("../data/hexagones_firepoint_3.geojson")
doubs_voisins = pd.read_csv("../data/data_train_test/doubs_tous_voisins.csv")
ain_voisins = pd.read_csv("../data/data_train_test/ain_tous_voisins.csv")
yvelines_voisins = pd.read_csv("../data/data_train_test/yvelines_tous_voisins.csv")
df = pd.read_csv('../data/firefrance/NATURELSfire.csv')

In [4]:
df.head()

,id_intervention,raison_sortie,date_debut,date_fin,FL_CCR_CCF,nb_eng_CC,label,year,geometry,latitude,longitude,IN,date,type,coef,h3
0,18BB000936,NATURELS,2018-01-05 17:41:19,2018-01-05 18:45:21,1.0,1.0,1.0,2018,POINT (5.40067950288725 46.11392703711171),46.113927,5.400680,True,2018-01-05,NATURELS,1.0,871f91c4dffffff
1,18BB001093,NATURELS,2018-01-06 17:56:49,2018-01-06 19:37:07,1.0,1.0,1.0,2018,POINT (5.341383006066522 46.14565445996581),46.145654,5.341383,True,2018-01-06,NATURELS,1.0,871f91c6effffff
2,18BB001862,NATURELS,2018-01-12 08:43:15,2018-01-12 09:36:25,1.0,1.0,1.0,2018,POINT (5.691933471524859 45.8654043668394),45.865404,5.691933,True,2018-01-12,NATURELS,1.0,871f910d6ffffff
3,18BB002367,NATURELS,2018-01-15 17:55:58,2018-01-15 18:59:14,1.0,1.0,1.0,2018,POINT (5.054655960570852 45.8729756400462),45.872976,5.054656,True,2018-01-15,NATURELS,1.0,871f91c9bffffff
4,18BB003056,NATURELS,2018-01-19 22:57:16,2018-01-20 00:21:51,1.0,1.0,1.0,2018,POINT (5.795944151313267 45.948246966190304),45.948247,5.795944,True,2018-01-19,NATURELS,1.0,871f910e3ffffff


In [5]:
df.columns

Index(['id_intervention', 'raison_sortie', 'date_debut', 'date_fin',
       'FL_CCR_CCF', 'nb_eng_CC', 'label', 'year', 'geometry', 'latitude',
       'longitude', 'IN', 'date', 'type', 'coef', 'h3'],
      dtype='object')